# Laboratorio 26. Calidad de datos con redes siamesas: detección de clientes duplicados

**Pregunta guía:** ¿Puede una red siamesa aprender a reconocer que dos registros con errores de tipeo, abreviaturas y campos faltantes corresponden al mismo cliente, mejor que una similitud de texto con umbral?

## Objetivos
Al finalizar podrás generar registros duplicados con errores realistas; particionar **por entidad** para evitar fuga; representar campos con n-gramas de caracteres; entrenar una red siamesa que compara pares; elegir umbrales con validación; usar los embeddings para reducir comparaciones (*blocking*); y estimar el número real de clientes únicos.

## Prerrequisitos y conexión
Usa embeddings (laboratorios 20 y 21) y n-gramas (laboratorio 25). Cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
Una red **siamesa** aplica el **mismo** codificador $f_\theta$ a dos entradas y compara sus representaciones: $P(\text{mismo}\mid a,b)=\sigma\big(g([\,|f_\theta(a)-f_\theta(b)|;\ f_\theta(a)\odot f_\theta(b)\,])\big)$. Compartir pesos garantiza que ambos registros se representen en el mismo espacio. Cada campo se describe por sus trigramas de caracteres ("rojas" → "#ro", "roj", "oja", "jas", "as#"), que toleran errores de tipeo; un `EmbeddingBag` promedia sus embeddings.

## Problema y datos
Un CRM ficticio contiene 2 000 clientes reales, varios registrados más de una vez con nombres abreviados ("Fco."), apodos, tildes faltantes, un apellido omitido, comunas escritas de distinta forma, correos faltantes o mal escritos y años de nacimiento digitados con error. Además existen homónimos (clientes distintos con el mismo nombre y apellido) y hermanos que comparten apellidos y comuna. Todos los datos se generan en el notebook.

## Perspectiva de negocio y datos
Los duplicados inflan el número de clientes, distorsionan métricas (CAC, *churn*, valor de vida) y generan comunicaciones repetidas. La resolución de entidades es la base del **registro maestro** (*golden record*) en un programa de gobierno de datos. Un falso positivo (fusionar dos personas distintas) es grave: puede mezclar datos personales de dos clientes.

## Experimento conceptual
¿Qué aporta aprender la similitud frente a definirla a mano? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional. Usamos `rng` (generador de NumPy) para los datos y `torch.manual_seed` antes de cada entrenamiento para que las comparaciones entre modelos sean justas.

In [ ]:
import copy
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 2026
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)

import itertools
import unicodedata
import zlib
from collections import defaultdict

**Resultado e interpretación.** La celda confirma el entorno utilizado. El laboratorio está diseñado para CPU y se ejecuta en pocos minutos; una GPU disponible se utiliza sin ser requisito.

### Generar clientes y registros duplicados
Cada cliente tiene un registro original y, con cierta probabilidad, de una a tres copias con errores.

In [ ]:
first_names = ['Francisco', 'José', 'María', 'Javiera', 'Camila', 'Felipe', 'Sebastián', 'Valentina', 'Catalina',
               'Matías', 'Benjamín', 'Constanza', 'Ignacio', 'Fernanda', 'Cristóbal', 'Daniela', 'Tomás', 'Antonia',
               'Joaquín', 'Carolina', 'Rodrigo', 'Paula', 'Andrés', 'Macarena', 'Diego', 'Francisca']
surnames = ['González', 'Muñoz', 'Rojas', 'Díaz', 'Pérez', 'Soto', 'Contreras', 'Silva', 'Martínez', 'Sepúlveda',
            'Morales', 'Rodríguez', 'López', 'Fuentes', 'Hernández', 'Torres', 'Araya', 'Flores', 'Espinoza',
            'Valenzuela', 'Castillo', 'Tapia', 'Reyes', 'Gutiérrez', 'Castro', 'Pizarro', 'Álvarez', 'Vásquez']
comunas = ['Santiago', 'Providencia', 'Ñuñoa', 'Las Condes', 'Maipú', 'Puente Alto', 'La Florida', 'Viña del Mar',
           'Valparaíso', 'Concepción', 'Temuco', 'Antofagasta', 'La Serena', 'Rancagua', 'Puerto Montt', 'Estación Central']
comuna_variants = {'Santiago': ['Stgo', 'Santiago Centro'], 'Viña del Mar': ['Vina del Mar', 'V. del Mar'],
                   'Estación Central': ['Est. Central'], 'Puerto Montt': ['Pto. Montt'], 'Ñuñoa': ['Nunoa'],
                   'Concepción': ['Conce'], 'Las Condes': ['Lascondes']}
nicknames = {'Francisco': ['Fco.', 'Pancho'], 'José': ['J.', 'Pepe'], 'María': ['Ma.'], 'Ignacio': ['Nacho'],
             'Sebastián': ['Seba'], 'Francisca': ['Fran'], 'Cristóbal': ['Cris'], 'Catalina': ['Cata'],
             'Constanza': ['Coni'], 'Fernanda': ['Feña']}


def strip_accents(text):
    return ''.join(ch for ch in unicodedata.normalize('NFKD', text) if not unicodedata.combining(ch))


def typo(text):
    if len(text) < 4:
        return text
    i = int(rng.integers(1, len(text) - 1))
    operation = rng.choice(['borrar', 'duplicar', 'intercambiar'])
    if operation == 'borrar':
        return text[:i] + text[i + 1:]
    if operation == 'duplicar':
        return text[:i] + text[i] + text[i:]
    return text[:i - 1] + text[i] + text[i - 1] + text[i + 1:]


n_entities = 2000
entities = []
for e in range(n_entities):
    first = str(rng.choice(first_names))
    domain = str(rng.choice(['gmail.com', 'hotmail.com', 'outlook.com', 'empresa.cl']))
    if e > 0 and rng.random() < 0.25:  # hermano o hermana de un cliente anterior: mismos apellidos y comuna
        sibling = entities[int(rng.integers(0, e))]
        s1, s2, comuna = sibling['s1'], sibling['s2'], sibling['comuna']
        year = int(sibling['anio']) + int(rng.choice([-1, 1])) * int(rng.integers(1, 8))
    else:
        s1, s2, comuna = str(rng.choice(surnames)), str(rng.choice(surnames)), str(rng.choice(comunas))
        year = int(rng.integers(1950, 2006))
    entities.append({'nombre': f'{first} {s1} {s2}', 'first': first, 's1': s1, 's2': s2, 'comuna': comuna,
                     'anio': str(year),
                     'email': f'{strip_accents(first).lower()}.{strip_accents(s1).lower()}{rng.integers(1, 99)}@{domain}'})


def corrupt(entity):
    first, s2 = entity['first'], entity['s2']
    if first in nicknames and rng.random() < 0.4:
        first = str(rng.choice(nicknames[first]))
    name = f'{first} {entity["s1"]}' + ('' if rng.random() < 0.3 else f' {s2}')
    if rng.random() < 0.3:
        name = typo(name)
    if rng.random() < 0.3:
        name = strip_accents(name)
    if rng.random() < 0.2:
        name = name.upper()
    comuna = entity['comuna']
    if comuna in comuna_variants and rng.random() < 0.4:
        comuna = str(rng.choice(comuna_variants[comuna]))
    elif rng.random() < 0.1:
        comuna = typo(comuna)
    email = '' if rng.random() < 0.4 else (typo(entity['email']) if rng.random() < 0.2 else entity['email'])
    anio = str(int(entity['anio']) + int(rng.choice([-1, 1]))) if rng.random() < 0.15 else entity['anio']
    return {'nombre': name, 'comuna': comuna, 'email': email, 'anio': anio}


records, record_entity = [], []
for e, entity in enumerate(entities):
    records.append({k: entity[k] for k in ['nombre', 'comuna', 'email', 'anio']})
    record_entity.append(e)
    for _ in range(int(rng.choice([0, 1, 2, 3], p=[0.40, 0.35, 0.15, 0.10]))):
        records.append(corrupt(entity))
        record_entity.append(e)
record_entity = np.array(record_entity)
print('clientes reales:', n_entities, '| registros en el CRM:', len(records),
      f'| tasa aparente de duplicación: {len(records) / n_entities - 1:.1%}')
for r in np.flatnonzero(record_entity == record_entity[np.argmax(np.bincount(record_entity))])[:4]:
    print('  ', records[r])

**Resultado e interpretación.** Los registros de un mismo cliente pueden no compartir ni un campo idéntico: el correo puede faltar, el nombre venir abreviado y la comuna escrita de otra forma. Al mismo tiempo, hay homónimos y hermanos que comparten varios campos sin ser la misma persona.

### Partición por entidad y construcción de pares
Separamos **clientes** (no registros) en train, validación y test: si las copias de un cliente quedaran repartidas entre train y test, el modelo memorizaría a esa persona. Por cada par positivo creamos un negativo difícil (un homónimo con el mismo nombre y primer apellido, o si no existe, un pariente con los mismos apellidos u otro cliente con el mismo primer apellido) y uno aleatorio.

In [ ]:
entity_order = rng.permutation(n_entities)
entity_split = {'train': entity_order[:1200], 'val': entity_order[1200:1600], 'test': entity_order[1600:]}


def build_pairs(entity_ids):
    recs = np.flatnonzero(np.isin(record_entity, entity_ids))
    by_entity = defaultdict(list)
    by_key = {'homonimo': defaultdict(list), 'pariente': defaultdict(list), 'apellido': defaultdict(list)}
    keys = {'homonimo': lambda x: (x['first'], x['s1']), 'pariente': lambda x: (x['s1'], x['s2']),
            'apellido': lambda x: x['s1']}
    for r in recs:
        by_entity[record_entity[r]].append(r)
        for kind, key in keys.items():
            by_key[kind][key(entities[record_entity[r]])].append(r)
    pairs, targets = [], []
    for e, rs in by_entity.items():
        for a, b in itertools.combinations(rs, 2):
            pairs.append((a, b)); targets.append(1)
            for kind, key in keys.items():  # el negativo más difícil disponible
                hard = [r for r in by_key[kind][key(entities[e])] if record_entity[r] != e]
                if hard:
                    pairs.append((a, int(rng.choice(hard)))); targets.append(0)
                    break
            other = int(rng.choice(recs))
            while record_entity[other] == e:
                other = int(rng.choice(recs))
            pairs.append((a, other)); targets.append(0)
    return np.array(pairs), np.array(targets, dtype=np.float32), recs


pair_data = {split: build_pairs(ids) for split, ids in entity_split.items()}
for split, (pairs, targets, recs) in pair_data.items():
    print(f'{split:5s} registros {len(recs):5d} | pares {len(pairs):5d} | positivos {int(targets.sum())}')

**Resultado e interpretación.** Hay aproximadamente dos negativos por positivo. En un CRM real la proporción es extrema (casi todos los pares posibles son negativos), por eso más adelante evaluamos también la búsqueda sobre todos los registros.

### Normalización y trigramas de caracteres
Normalizamos (minúsculas, sin tildes, espacios simples) y convertimos cada campo en una lista de trigramas codificados por *hashing*. Un campo vacío queda solo con relleno.

In [ ]:
FIELDS = {'nombre': 40, 'comuna': 20, 'email': 40, 'anio': 6}  # largo máximo en trigramas
N_BUCKETS = 5000


def normalize(text):
    return ' '.join(strip_accents(text).lower().split())


def trigram_ids(text, max_len):
    text = normalize(text)
    if not text:
        return [0] * max_len
    padded = f'#{text}#'
    ids = [zlib.crc32(padded[i:i + 3].encode()) % N_BUCKETS + 1 for i in range(len(padded) - 2)][:max_len]
    return ids + [0] * (max_len - len(ids))


encoded = {field: torch.tensor([trigram_ids(r[field], length) for r in records], dtype=torch.long, device=device)
           for field, length in FIELDS.items()}


def gather(record_ids):
    return {field: encoded[field][record_ids] for field in FIELDS}


print('ejemplo:', records[0]['nombre'], '->', encoded['nombre'][0][:8].tolist(), '...')

**Resultado e interpretación.** "Rojas" y "Rojs" comparten varios trigramas aunque las palabras no sean iguales: esa superposición es la que el modelo puede aprovechar. El *hashing* fija el tamaño de la tabla sin construir un vocabulario, a costa de colisiones ocasionales.

### Baseline: similitud de Jaccard con umbral
La similitud de Jaccard entre los conjuntos de trigramas de los dos registros completos es $|A\cap B|/|A\cup B|$. Elegimos el umbral que maximiza F1 en validación. También medimos la regla "mismo correo".

In [ ]:
record_grams = [set(g for field, length in FIELDS.items() for g in trigram_ids(r[field], length) if g) for r in records]


def jaccard(a, b):
    return len(record_grams[a] & record_grams[b]) / max(len(record_grams[a] | record_grams[b]), 1)


def pair_metrics(targets, predicted):
    tp = float(((predicted == 1) & (targets == 1)).sum())
    precision = tp / max(predicted.sum(), 1)
    recall = tp / max(targets.sum(), 1)
    return {'precision': round(float(precision), 4), 'recall': round(float(recall), 4),
            'f1': round(float(2 * precision * recall / max(precision + recall, 1e-9)), 4)}


def best_threshold(targets, scores):
    grid = np.linspace(0.05, 0.95, 91)
    f1 = [pair_metrics(targets, (scores >= t).astype(int))['f1'] for t in grid]
    return float(grid[int(np.argmax(f1))])


jaccard_scores = {split: np.array([jaccard(a, b) for a, b in pairs]) for split, (pairs, _, _) in pair_data.items()}
jaccard_threshold = best_threshold(pair_data['val'][1], jaccard_scores['val'])
print(f'Jaccard (umbral {jaccard_threshold:.2f}) validación:',
      pair_metrics(pair_data['val'][1], (jaccard_scores['val'] >= jaccard_threshold).astype(int)))
same_email = np.array([records[a]['email'] != '' and records[a]['email'] == records[b]['email']
                       for a, b in pair_data['val'][0]]).astype(int)
print('regla "mismo correo" validación:', pair_metrics(pair_data['val'][1], same_email))

**Resultado e interpretación.** La regla del correo tiene precisión alta pero *recall* bajo: falla cuando el correo falta o tiene errores. Jaccard trata todos los trigramas por igual; no sabe que coincidir en el año pesa poco y que un apellido raro pesa mucho.

### Red siamesa
El codificador tiene un `EmbeddingBag` por campo y una proyección común; su salida se normaliza. La cabeza compara las dos representaciones con $|a-b|$ y $a\odot b$.

In [ ]:
class RecordEncoder(nn.Module):
    def __init__(self, dim=32, out_dim=64):
        super().__init__()
        self.bags = nn.ModuleDict({field: nn.EmbeddingBag(N_BUCKETS + 1, dim, mode='mean', padding_idx=0)
                                   for field in FIELDS})
        self.project = nn.Sequential(nn.Linear(dim * len(FIELDS), 128), nn.ReLU(), nn.Linear(128, out_dim))

    def forward(self, fields):
        h = torch.cat([self.bags[field](fields[field]) for field in FIELDS], dim=1)
        return F.normalize(self.project(h), dim=1)


class SiameseMatcher(nn.Module):
    def __init__(self, out_dim=64):
        super().__init__()
        self.encoder = RecordEncoder(out_dim=out_dim)  # el mismo codificador para ambos registros
        self.head = nn.Sequential(nn.Linear(2 * out_dim, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, a, b):
        ea, eb = self.encoder(a), self.encoder(b)
        return self.head(torch.cat([(ea - eb).abs(), ea * eb], dim=1)).squeeze(1)


pair_tensors = {split: (torch.tensor(pairs, dtype=torch.long, device=device),
                        torch.tensor(targets, dtype=torch.float32, device=device))
                for split, (pairs, targets, _) in pair_data.items()}
print('parámetros:', sum(p.numel() for p in SiameseMatcher().parameters()))

**Resultado e interpretación.** La simetría importa: con $|a-b|$ y $a\odot b$, el puntaje del par (a, b) es igual al de (b, a). Casi todos los parámetros están en las tablas de trigramas.

### Entrenar la red siamesa
Iteramos sobre índices de pares y reunimos los campos de cada registro en el batch.

In [ ]:
def match_logits(model, pairs):
    return model(gather(pairs[:, 0]), gather(pairs[:, 1]))


def train_matcher(epochs=40, lr=2e-3, batch_size=256, patience=6):
    torch.manual_seed(SEED)
    model = SiameseMatcher().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_pairs, train_targets = pair_tensors['train']
    loader = DataLoader(TensorDataset(torch.arange(len(train_targets))), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    curve, best_loss, best_state, wait = [], np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        for (batch,) in loader:
            batch = batch.to(device)
            loss = F.binary_cross_entropy_with_logits(match_logits(model, train_pairs[batch]), train_targets[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.binary_cross_entropy_with_logits(match_logits(model, pair_tensors['val'][0]),
                                                          pair_tensors['val'][1]).item()
        curve.append(val_loss)
        if val_loss < best_loss - 1e-4:
            best_loss, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, curve


start = time.perf_counter()
matcher, curve = train_matcher()
print(f'{len(curve)} épocas | {time.perf_counter() - start:.1f} s')


def match_proba(split):
    matcher.eval()
    with torch.no_grad():
        return torch.sigmoid(match_logits(matcher, pair_tensors[split][0])).cpu().numpy()


siamese_threshold = best_threshold(pair_data['val'][1], match_proba('val'))
print(f'siamesa (umbral {siamese_threshold:.2f}) validación:',
      pair_metrics(pair_data['val'][1], (match_proba('val') >= siamese_threshold).astype(int)))
print('\nTest (umbrales elegidos en validación):')
print('  Jaccard:', pair_metrics(pair_data['test'][1], (jaccard_scores['test'] >= jaccard_threshold).astype(int)))
print('  siamesa:', pair_metrics(pair_data['test'][1], (match_proba('test') >= siamese_threshold).astype(int)))

**Resultado e interpretación.** Compara precisión y *recall* por separado. Si la siamesa mejora sobre todo el *recall*, aprendió a tolerar abreviaturas, apodos y campos faltantes; si mejora la precisión, aprendió a distinguir homónimos usando año, comuna y segundo apellido. Recuerda que fusionar registros por error (falso positivo) suele ser más costoso que no fusionarlos.

### Embeddings para buscar candidatos (*blocking*)
Comparar todos los pares de un CRM con $n$ registros requiere $n(n-1)/2$ evaluaciones. Con los embeddings del codificador buscamos los $k$ vecinos más cercanos de cada registro y solo comparamos esos. Medimos qué fracción de los registros con duplicado lo encuentra entre sus vecinos.

In [ ]:
test_records = pair_data['test'][2]
matcher.eval()
with torch.no_grad():
    embeddings = matcher.encoder(gather(torch.tensor(test_records, device=device))).cpu().numpy()
similarity = embeddings @ embeddings.T
np.fill_diagonal(similarity, -np.inf)
neighbors = np.argsort(-similarity, axis=1)
test_entity = record_entity[test_records]
has_duplicate = np.array([(test_entity == e).sum() > 1 for e in test_entity])
for k in (1, 5, 10):
    found = (test_entity[neighbors[:, :k]] == test_entity[:, None]).any(axis=1)
    print(f'recall@{k:2d}: {found[has_duplicate].mean():.3f} | comparaciones: {len(test_records) * k} '
          f'en vez de {len(test_records) * (len(test_records) - 1) // 2}')

**Resultado e interpretación.** Si el *recall*@5 es alto, basta comparar cinco candidatos por registro en lugar de todos contra todos: el costo pasa de cuadrático a lineal. Esta es la estrategia usual en resolución de entidades a gran escala.

### Estimar clientes únicos: un indicador de calidad de datos
Unimos registros cuando la siamesa supera el umbral entre un registro y alguno de sus 5 vecinos, y agrupamos por componentes conectadas (*union-find*). Comparamos el número estimado de clientes únicos con el real del set de test.

In [ ]:
parent = list(range(len(test_records)))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


candidates = np.array(sorted({(min(i, int(j)), max(i, int(j))) for i in range(len(test_records)) for j in neighbors[i, :5]}))
with torch.no_grad():
    candidate_ids = torch.tensor(test_records[candidates], dtype=torch.long, device=device)
    candidate_proba = torch.sigmoid(match_logits(matcher, candidate_ids)).cpu().numpy()
for (i, j), p in zip(candidates, candidate_proba):
    if p >= siamese_threshold:
        parent[find(i)] = find(j)

clusters = np.array([find(i) for i in range(len(test_records))])
wrong_merges = sum(len(set(test_entity[clusters == c])) > 1 for c in np.unique(clusters))
print('registros en test:', len(test_records))
print('clientes únicos reales:', len(np.unique(test_entity)), '| estimados:', len(np.unique(clusters)))
print('grupos que mezclan clientes distintos:', wrong_merges)

**Resultado e interpretación.** La diferencia entre registros y clientes estimados es el indicador de duplicación que reportaría la oficina del CDO. Los grupos que mezclan clientes distintos son el riesgo principal: en producción, las fusiones dudosas se envían a revisión humana antes de modificar el registro maestro.

## Limitaciones y conclusiones clave
Los errores se generaron con reglas conocidas; en un CRM real aparecen patrones nuevos (cambios de apellido, direcciones, empresas) y las etiquetas de pares se obtienen de revisiones manuales. Las fusiones deben ser auditables y reversibles, y los datos personales usados para comparar deben tratarse con las mismas reglas de acceso que el CRM.

- Una red siamesa aprende una función de similitud compartiendo el codificador entre ambos registros.
- La partición por entidad y los negativos difíciles (homónimos) son esenciales para una evaluación honesta.
- Los embeddings permiten *blocking* eficiente y la agrupación produce un indicador de calidad de datos.

## Ejercicios progresivos
1. **Guiado:** quita los negativos difíciles de `build_pairs` (solo negativos aleatorios), reentrena y compara la precisión en test, que sí contiene homónimos. Explica la diferencia.
2. **Independiente:** elimina el campo `email` de `FIELDS` y mide cuánto caen precisión y *recall*. ¿Qué campo aporta más a cada métrica?
3. **Desafío:** cambia el criterio de umbral para exigir precisión de al menos 0,99 en validación (en vez de maximizar F1) y repite la estimación de clientes únicos. Discute el costo de no fusionar duplicados frente al de fusionar personas distintas.

## Conexión con el siguiente laboratorio
Con estos laboratorios cubriste visión, anomalías, texto y calidad de datos. Combínalos en un proyecto con datos reales de tu organización, documentando origen, calidad, licencia y tratamiento de datos personales.